In [ ]:
import nltk
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding,LSTM
from tensorflow.keras.optimizers import SGD
from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
nltk.download('movie_reviews')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\visha\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


True

In [ ]:
import pandas as pd

data = [(movie_reviews.raw(fileid), category) for category in movie_reviews.categories() for fileid in movie_reviews.fileids(category)]

documents, labels = [data[i][0] for i in range(len(data))], [data[i][1] for i in range(len(data))]
df=pd.DataFrame()
df["review"]=documents
df["label"]=labels
df

,review,label
0,"plot : two teen couples go to a church party ,...",neg
1,the happy bastard's quick movie review \ndamn ...,neg
2,it is movies like these that make a jaded movi...,neg
3,""" quest for camelot "" is warner bros . ' firs...",neg
4,synopsis : a mentally unstable man undergoing ...,neg
...,...,...
1995,wow ! what a movie . \nit's everything a movie...,pos
1996,"richard gere can be a commanding actor , but h...",pos
1997,"glory--starring matthew broderick , denzel was...",pos
1998,steven spielberg's second epic film on world w...,pos


In [ ]:
from sklearn.preprocessing import LabelEncoder
le= LabelEncoder()
labels = le.fit_transform(labels)

In [ ]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

# nltk.download('movie_reviews')
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')

#Lower case
def lower(text):
    return text.lower()

#Punctuation remove kr
def punct_remove(text):
    new_text = re.sub(r'[^\w\s]', '', text)
    new_text = re.sub(r'https?://\S+', '',new_text)
    return new_text

def tokenize(text):
    return word_tokenize(text)

def stop_word_remove(tokens):
    stop_words = set(stopwords.words('english'))
    result = [word for word in tokens if word not in stop_words and word.isalpha()]
    return result

def stemming(tokens):
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(word) for word in tokens]
    return stemmed_tokens

def lemmatizing(tokens):
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return lemmatized_tokens

def preprocess_text(text):

    #lower the text
    text_lower = lower(text)

    #remove punctuation
    punct_text = punct_remove(text_lower)

    #tokenize text
    tokens = tokenize(punct_text)

     #remove stopwords
    tokens = stop_word_remove(tokens)

    stemmed_tokens = stemming(tokens)   #stemmetize

    lemmatized_tokens = lemmatizing(tokens) #lemmetize

    return " ".join(lemmatized_tokens)

demo = "This is a boy girl"
preprocess_text(demo)

'boy girl'

In [ ]:
df["review"]=df["review"].apply(preprocess_text)
df

,review,label
0,plot two teen couple go church party drink dri...,neg
1,happy bastard quick movie review damn bug got ...,neg
2,movie like make jaded movie viewer thankful in...,neg
3,quest camelot warner bros first featurelength ...,neg
4,synopsis mentally unstable man undergoing psyc...,neg
...,...,...
1995,wow movie everything movie funny dramatic inte...,pos
1996,richard gere commanding actor he always great ...,pos
1997,glorystarring matthew broderick denzel washing...,pos
1998,steven spielberg second epic film world war ii...,pos


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
documents = df["review"].values

cv=CountVectorizer() #to count unique words
cv.fit(documents)
vocab_size = len(cv.vocabulary_)
print(vocab_size)
print(cv.vocabulary_)
cv.get_feature_names_out()

41380
{'plot': 27481, 'two': 38074, 'teen': 36423, 'couple': 7641, 'go': 14833, 'church': 6018, 'party': 26493, 'drink': 10423, 'drive': 10431, 'get': 14551, 'accident': 172, 'one': 25453, 'guy': 15592, 'dy': 10686, 'girlfriend': 14674, 'continues': 7274, 'see': 32158, 'life': 20782, 'nightmare': 24547, 'whats': 40311, 'deal': 8553, 'watch': 39970, 'movie': 23666, 'sorta': 34162, 'find': 13080, 'critique': 7920, 'mindfuck': 22980, 'generation': 14437, 'touch': 37403, 'cool': 7377, 'idea': 17465, 'present': 28149, 'bad': 2440, 'package': 26194, 'make': 21734, 'review': 30500, 'even': 11871, 'harder': 15947, 'write': 40932, 'since': 33251, 'generally': 14432, 'applaud': 1561, 'film': 13015, 'attempt': 2091, 'break': 4217, 'mold': 23317, 'mess': 22725, 'head': 16169, 'lost': 21268, 'highway': 16629, 'memento': 22607, 'good': 14955, 'way': 40017, 'making': 21743, 'type': 38120, 'folk': 13501, 'didnt': 9359, 'snag': 33813, 'correctly': 7510, 'seem': 32168, 'taken': 36160, 'pretty': 28204, '

array(['aa', 'aaa', 'aaaaaaaaah', ..., 'zwicks', 'zwigoffs', 'zycie'],
      dtype=object)

In [ ]:
max_length = max([len(i) for i in documents])
max_length

10320

In [ ]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(documents)

sequences = tokenizer.texts_to_sequences(documents)

max_length = 1000
X = pad_sequences(sequences, maxlen=max_length, padding='post')
print(X.shape)
print(X[0])

(2000, 1000)
[  27   18  701  214   21 1657  581 2587  835    7 1104    3   78 1164
  520 1400   16   19 1452  480  382  131    2 8380   55 1908    2  701
 1006  725  587  175  437   33 3211    9  266   10 3285    3 1088  105
 1186 7818    1  195  476 5312  732  220  252 3561 7349   11   33   17
  154  509    1 1176  150    3 5518  151  504  174 3466  962 2643 2107
  119    2   22  253  119  232 6964  136 1128  885   66   57  407  175
  480   82  391    4  425   62  291  360   41    5  291  634 9776 9011
  568    8 2993 1276   23  516  232 1744 2588   46  196  157 8381    1
   73   58 1582    7  146 3467    1  812  119  503  193   71  536 2021
   64   65 2021  223  271  450  107    9   23  341 2542   10 1583 5081
   26 1142   76 3787 3667  408    5   88 3668 1967   89  136    9   29
  139  162   63  150    9    1  341  490 1484  113    2    5  152    9
  179   57   10  190  536 1796   66 2073  248  798 6965 9012  340  127
 1277  107  367    2 1503 3286  980    7   31 2840   46   30   2

In [ ]:
tokenizer.word_index

{'film': 1,
 'movie': 2,
 'one': 3,
 'character': 4,
 'like': 5,
 'time': 6,
 'get': 7,
 'scene': 8,
 'make': 9,
 'even': 10,
 'good': 11,
 'story': 12,
 'would': 13,
 'much': 14,
 'also': 15,
 'see': 16,
 'way': 17,
 'two': 18,
 'life': 19,
 'first': 20,
 'go': 21,
 'well': 22,
 'thing': 23,
 'year': 24,
 'take': 25,
 'really': 26,
 'plot': 27,
 'come': 28,
 'little': 29,
 'know': 30,
 'people': 31,
 'could': 32,
 'bad': 33,
 'work': 34,
 'never': 35,
 'man': 36,
 'performance': 37,
 'end': 38,
 'best': 39,
 'new': 40,
 'look': 41,
 'doesnt': 42,
 'many': 43,
 'actor': 44,
 'director': 45,
 'dont': 46,
 'play': 47,
 'u': 48,
 'love': 49,
 'action': 50,
 'he': 51,
 'role': 52,
 'great': 53,
 'show': 54,
 'find': 55,
 'another': 56,
 'audience': 57,
 'give': 58,
 'star': 59,
 'say': 60,
 'something': 61,
 'back': 62,
 'still': 63,
 'seems': 64,
 'want': 65,
 'world': 66,
 'made': 67,
 'there': 68,
 'however': 69,
 'think': 70,
 'big': 71,
 'day': 72,
 'every': 73,
 'though': 74,
 'bette

In [ ]:
a = ["my name is vishal"]
tokenizer.texts_to_sequences(a)

[[204]]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)
embedding_dim = 64
# X_train_oh = tf.one_hot(X_train, depth=vocab_size)
# X_test_oh = tf.one_hot(X_test, depth=vocab_size)
# X_train_oh.shape

In [ ]:
model = Sequential([
    Embedding(input_dim = vocab_size,output_dim = embedding_dim, input_length = max_length),
    SimpleRNN(128, return_sequences=False, activation='relu'),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 1000, 64)          640000    
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               24704     
                                                                 
 dense (Dense)               (None, 128)               16512     
                                                                 
 dense_1 (Dense)             (None, 1)                 129       
                                                                 
Total params: 681345 (2.60 MB)
Trainable params: 681345 (2.60 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
opt = SGD(learning_rate=0.1)
model.compile(optimizer=opt,loss='binary_crossentropy',metrics=['accuracy'])

model.fit(X_train, y_train, epochs=15, batch_size=32, validation_split=0.2)

Epoch 1/15


40/40 [==============================] - 16s 314ms/step - loss: 0.6934 - accuracy: 0.4992 - val_loss: 0.6977 - val_accuracy: 0.4563
Epoch 2/15
14/40 [=========>....................] - ETA: 6s - loss: 0.6950 - accuracy: 0.4866

KeyboardInterrupt: 

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)